# 08 — PM2.5-radiative shading mechanism test (Bogotá, O3)

Decisive test for the 2026-09-10 TODO ("Take the PM2.5-radiative effect on O3 seriously"):
rather than pre-committing to a functional form (the original `radiation_mean/(1+pm25)`
reciprocal, or a manually-swept `radiation_mean*exp(-k*pm25)`), let **PySR search freely**
over `exp`/`log`/`sqrt` combinations of the raw `pm25` and `radiation_mean` variables
(same-day, and separately with `pm25_lag1`) to see what functional form and effective
extinction coefficient it actually settles on, unconstrained by our own guess at the
right form.

If PySR converges on something resembling `radiation_mean * exp(-k*pm25)`, the fitted
`k` can be compared against literature PM2.5 mass-extinction efficiencies (~3-4 m^2/g)
as an independent physical-plausibility check on whether this is a real optical
mechanism or a correlation of convenience -- see the k-sweep already logged, which
crossed zero around k=0.025-0.03 in raw units (not yet converted to physical m^2/g,
since the pm25 units and PySR's ability to encode an implicit unit-conversion constant
both matter for that final step).


In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

df = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")
df = df.dropna(subset=['o3']).reset_index(drop=True)
if 'pm25_lag1' not in df.columns:
    df['pm25_lag1'] = df['pm25'].shift(1)
print(df.shape)


(3285, 215)


## Run 1: same-day `pm25` + `radiation_mean`, free search

In [2]:
from pysr import PySRRegressor

features_same = ['pm25', 'radiation_mean']
data_same = df.dropna(subset=features_same + ['o3']).reset_index(drop=True)
X_same = data_same[features_same].to_numpy()
y_same = data_same['o3'].to_numpy()
print(data_same.shape)

sr_same = PySRRegressor(
    niterations=150,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["exp", "log", "sqrt"],
    model_selection="best",
    maxsize=20,
    populations=24,
)
sr_same.fit(X_same, y_same, variable_names=features_same)
print(sr_same)


Compiling Julia backend...


(3285, 215)


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.350e+04
Progress: 63 / 150 total iterations (42.056%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.794e+01  0.000e+00  y = 20.22
3           4.024e+01  8.747e-02  y = radiation_mean * 0.10589
4           3.773e+01  6.439e-02  y = log(pm25) * 7.1948
5           3.658e+01  3.096e-02  y = (pm25 * 0.42482) + 12.348
6           3.171e+01  1.430e-01  y = sqrt(radiation_mean) + (pm25 * 0.36344)
8           3.156e+01  2.389e-03  y = (pm25 * 0.39474) + sqrt(radiation_mean - pm25)
10          3.145e+01  1.711e-03  y = ((pm25 * 0.35664) - -1.6036) + sqrt(radiation_mean + -...
                                      39.032)
11          3.115e+01  9.459e-03  y = sqrt(radiation_mean) + ((pm25 + (sqrt(radiation_mean) ...
                                    

PySRRegressor.equations_ = [
	   pick     score                                           equation  \
	0        0.000000                                          20.218948   
	1        0.087472                       radiation_mean * 0.105726704   
	2        0.064391                               log(pm25) * 7.194789   
	3        0.042404               (radiation_mean + pm25) * 0.09703713   
	4  >>>>  0.131596         sqrt(radiation_mean) + (pm25 * 0.36344182)   
	5        0.012614  ((pm25 * 5.4480834) + radiation_mean) * 0.0694...   
	6        0.012029  ((pm25 * 0.023669569) - -1.0253984) * sqrt(rad...   
	7        0.009468  (((pm25 + 15.093067) * radiation_mean) * 0.001...   
	8        0.000005  (((radiation_mean + 1.6859908) * 0.0017350045)...   
	9        0.000171  (0.0017350045 * ((radiation_mean + (pm25 / 14....   
	
	        loss  complexity  
	0  47.937557           1  
	1  40.243760           3  
	2  37.734110           4  
	3  36.167470           5  
	4  31.707834           6 

In [3]:
pareto_same = sr_same.equations_[['complexity', 'loss', 'equation']]
pareto_same['r2'] = pareto_same.apply(
    lambda row: r2_score(y_same, sr_same.predict(X_same, index=row.name)), axis=1
)
pareto_same


,complexity,loss,equation,r2
0,1,47.937557,20.218948,-1.339717e-07
1,3,40.243760,radiation_mean * 0.105726704,1.604961e-01
2,4,37.734110,log(pm25) * 7.194789,2.128484e-01
3,5,36.167470,(radiation_mean + pm25) * 0.09703713,2.455294e-01
4,6,31.707834,sqrt(radiation_mean) + (pm25 * 0.36344182),3.385596e-01
5,7,31.310383,((pm25 * 5.4480834) + radiation_mean) * 0.0694...,3.468505e-01
6,8,30.936020,((pm25 * 0.023669569) - -1.0253984) * sqrt(rad...,3.546599e-01
7,9,30.644491,(((pm25 + 15.093067) * radiation_mean) * 0.001...,3.607414e-01
8,11,30.644169,(((radiation_mean + 1.6859908) * 0.0017350045)...,3.607481e-01
9,13,30.633696,(0.0017350045 * ((radiation_mean + (pm25 / 14....,3.609666e-01


## Run 2: `pm25_lag1` (yesterday's PM2.5) + `radiation_mean`, free search

In [4]:
features_lag = ['pm25_lag1', 'radiation_mean']
data_lag = df.dropna(subset=features_lag + ['o3']).reset_index(drop=True)
X_lag = data_lag[features_lag].to_numpy()
y_lag = data_lag['o3'].to_numpy()
print(data_lag.shape)

sr_lag = PySRRegressor(
    niterations=150,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["exp", "log", "sqrt"],
    model_selection="best",
    maxsize=20,
    populations=24,
)
sr_lag.fit(X_lag, y_lag, variable_names=features_lag)
print(sr_lag)


(3284, 215)


Expressions evaluated per second: 2.560e+04
Progress: 66 / 150 total iterations (44.111%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.795e+01  0.000e+00  y = 20.212
3           4.026e+01  8.747e-02  y = radiation_mean * 0.1058
4           3.898e+01  3.217e-02  y = sqrt(radiation_mean * 2.2206)
5           3.618e+01  7.444e-02  y = (radiation_mean + pm25_lag1) * 0.096106
6           3.418e+01  5.702e-02  y = log(sqrt(pm25_lag1)) * sqrt(radiation_mean)
7           3.277e+01  4.198e-02  y = sqrt(radiation_mean / sqrt(3.753 / pm25_lag1))
8           3.217e+01  1.872e-02  y = sqrt((pm25_lag1 + 13.258) * (radiation_mean / 14.032))
9           3.194e+01  7.123e-03  y = sqrt(((pm25_lag1 + sqrt(radiation_mean)) / 14.155) * r...
                                      adiation_mean)
11  

[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


In [5]:
pareto_lag = sr_lag.equations_[['complexity', 'loss', 'equation']]
pareto_lag['r2'] = pareto_lag.apply(
    lambda row: r2_score(y_lag, sr_lag.predict(X_lag, index=row.name)), axis=1
)
pareto_lag


,complexity,loss,equation,r2
0,1,47.951030,20.217146,-3.597447e-08
1,3,40.255043,radiation_mean * 0.10579874,1.604968e-01
2,4,38.980522,sqrt(radiation_mean * 2.2206106),1.870765e-01
3,5,36.147840,(radiation_mean + pm25_lag1) * 0.097682856,2.461510e-01
4,6,33.467937,(pm25_lag1 * 0.34759063) + sqrt(radiation_mean),3.020393e-01
5,7,32.031216,(pm25_lag1 / 3.2136478) + (radiation_mean * 0....,3.320015e-01
6,9,31.779722,((radiation_mean * 0.07008427) + 1.3265139) + ...,3.372463e-01
7,11,31.769684,(((radiation_mean * 0.0688996) + 0.9095698) + ...,3.374557e-01
8,13,31.759577,(pm25_lag1 / 3.2136478) + ((0.07008427 * radia...,3.376664e-01
9,16,31.707130,(pm25_lag1 / 3.4114463) + (((radiation_mean * ...,3.387602e-01


## Interpretation

Check whether either Pareto front contains an `exp(-k*pm25)`-shaped term (or any other
`exp`/`log` combination of the two raw variables), and if so, what R2 it achieves versus
the `radiation_mean`-only baseline (already known: raw corr = 0.450, i.e. R2 ~ 0.20) and
versus the already-tried manual forms (reciprocal R2 ~0.03, exponential sweep peak R2 ~
0.20-0.34 in the 0.28-0.37 correlation range translated to R2). A genuine, PySR-discovered
`exp` term with R2 clearly above the `radiation_mean`-only baseline is real independent
evidence for the mechanism; if PySR's best equations don't use `exp` at all (or barely
improve over radiation-alone), that itself is evidence against the shading hypothesis --
report either outcome honestly.


In [6]:
baseline_r2_same = r2_score(y_same, np.polyval(np.polyfit(data_same['radiation_mean'], y_same, 1), data_same['radiation_mean']))
print(f"radiation_mean-only linear baseline R2 (same-day subset): {baseline_r2_same:.3f}")
baseline_r2_lag = r2_score(y_lag, np.polyval(np.polyfit(data_lag['radiation_mean'], y_lag, 1), data_lag['radiation_mean']))
print(f"radiation_mean-only linear baseline R2 (lag subset):      {baseline_r2_lag:.3f}")

print()
print("Best (highest-R2) same-day equation:")
print(pareto_same.loc[pareto_same['r2'].idxmax()])
print()
print("Best (highest-R2) lag equation:")
print(pareto_lag.loc[pareto_lag['r2'].idxmax()])


radiation_mean-only linear baseline R2 (same-day subset): 0.202
radiation_mean-only linear baseline R2 (lag subset):      0.202

Best (highest-R2) same-day equation:
complexity                                                   13
loss                                                  30.633696
equation      (0.0017350045 * ((radiation_mean + (pm25 / 14....
r2                                                     0.360967
Name: 9, dtype: object

Best (highest-R2) lag equation:
complexity                                                   16
loss                                                   31.70713
equation      (pm25_lag1 / 3.4114463) + (((radiation_mean * ...
r2                                                      0.33876
Name: 9, dtype: object


## Save

In [7]:
pareto_same.to_csv(DATA_DIR / "bogota_o3_pm25_shading_test_sameday_pareto.csv", index=False)
pareto_lag.to_csv(DATA_DIR / "bogota_o3_pm25_shading_test_lag_pareto.csv", index=False)
print("Saved both Pareto fronts")


Saved both Pareto fronts
